# 12 — Per-acquisition Sentinel-1 and daily rainfall, 2022–2025

**Status: SOURCE WORKFLOW ONLY — not executed or validated.** Running this notebook requires Earth Engine access and a local project boundary. It exports actual results only after successful extraction; there are no invented per-scene results in the committed file.

Study question: can individual Sentinel-1 scenes distinguish the 2022, 2024 and 2025 high-June screening extents (with 2023 as a low-June comparator) when monthly medians cannot?

- Compare Sentinel-1 ascending orbit 41 and descending orbit 77 **separately**; never combine VV intensities as if geometry is equivalent.
- Per-scene fixed −19/−17/−15 dB VV thresholds, incidence-angle screen, 30 m focal median with ORIGINAL image support mask preserved.
- Report acquisition timestamps in UTC and Asia/Dhaka. CHIRPS daily image DATE LABELS are not assumed to equal Bangladesh midnight-to-midnight rainfall totals. Rainfall-date offsets of −1/0/+1 day are sensitivity alternatives, not corrections.
- Area is LOW-VV SCREENING, not validated water extent, depth, damage or flood severity. First/last observed detections do not resolve onset or recession between passes.
- Inputs: study-area boundary, original Notebook 09 scene inventory, Earth Engine Sentinel-1 GRD and CHIRPS daily. Outputs: CSVs, figures, QA report and optional separate per-orbit frequency maps.


## 12A — Configuration, boundary, Earth Engine and reproducibility

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import ee

start = Path(os.environ.get("HAKALUKI_ROOT", Path.cwd())).expanduser().resolve()
for candidate in [start, *start.parents]:
    if ((candidate/"data/boundary/hakaluki_boundary.gpkg").is_file()
        or (candidate/"Study area/Hakaluki_Haor.shp").is_file()):
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Set HAKALUKI_ROOT to the research project or start Jupyter inside it")
sys.path.insert(0, str(ROOT))
from src.scene12_utils import normalize_daily_rain, attach_rain_to_scenes, check_scene_areas, describe_scene_support

BOUNDARY = ROOT/"data/boundary/hakaluki_boundary.gpkg"
if not BOUNDARY.is_file():
    BOUNDARY = ROOT/"Study area/Hakaluki_Haor.shp"
TABLES, FIGURES, CACHE = ROOT/"results/tables", ROOT/"results/figures", ROOT/"data/processed/scene12_cache"
for folder in (TABLES, FIGURES, CACHE):
    folder.mkdir(parents=True, exist_ok=True)

YEARS = (2022, 2023, 2024, 2025)
MONTHS = (5, 9)             # May through September inclusive
TRACKS = (("ASCENDING", 41), ("DESCENDING", 77))
THRESHOLDS_DB = (-19.0, -17.0, -15.0)
ANGLE_RANGE = (30.0, 45.0)
FILTER_RADIUS_M, SCALE_M = 30, 30
MIN_SCENE_COVERAGE = 0.90
FORCE_RECOMPUTE = False
DO_VV_VH_SAMPLES = True

haor = gpd.read_file(BOUNDARY).to_crs("EPSG:4326")
assert len(haor) and haor.geometry.notna().all() and haor.geometry.is_valid.all()
polygon = haor.geometry.union_all()
project = os.getenv("EE_PROJECT") or os.getenv("GOOGLE_CLOUD_PROJECT")
try:
    if project:
        ee.Initialize(project=project)
    else:
        ee.Initialize()
except Exception as err:
    raise RuntimeError("Initialize Earth Engine and set EE_PROJECT locally if required; do not publish credentials") from err
ROI = ee.Geometry(polygon.__geo_interface__)
roi_stats = ee.Image.pixelArea().rename("roi_m2").reduceRegion(
    reducer=ee.Reducer.sum(), geometry=ROI, crs="EPSG:32646",
    scale=SCALE_M, maxPixels=100_000_000, tileScale=4).getInfo()
REFERENCE_KM2 = float(roi_stats["roi_m2"])/1e6
assert 10 < REFERENCE_KM2 < 500

CONFIG = {"years": list(YEARS), "months": list(MONTHS), "tracks": [list(x) for x in TRACKS],
          "thresholds": list(THRESHOLDS_DB), "angle": list(ANGLE_RANGE),
          "radius_m": FILTER_RADIUS_M, "scale_m": SCALE_M,
          "roi_wkb_hash": hashlib.sha256(polygon.wkb).hexdigest(),
          "source": "COPERNICUS/S1_GRD", "version": "12.1"}
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:12]
print("Project folder:", ROOT, "| EE reference area:", round(REFERENCE_KM2,3), "km²")
print("Configuration hash:", CONFIG_HASH, " | no new result until executed")


## 12B — Cross-check exact scene identities and acquisition times

The local Notebook 09 inventory must agree with the current Earth Engine catalog for both target orbits and every year. Mismatches abort the extraction rather than silently changing the sample. Local civil time conversion uses the named Asia/Dhaka timezone.

In [ ]:
candidate_inventories = [
    ROOT/"results/tables/inundation09_s1_scene_inventory.csv",
    ROOT/"docs/assets/tables/analysis_2019_2025/inundation09_s1_scene_inventory.csv"
]
inventory_path = next((p for p in candidate_inventories if p.exists()), None)
if inventory_path is None:
    raise FileNotFoundError("Run Notebook 09 first or copy its scene inventory into docs/assets/tables")
inventory = pd.read_csv(inventory_path)
necessary = {"scene_id","acquisition_utc","pass","relative_orbit"}
assert necessary.issubset(inventory.columns), f"Missing inventory columns: {necessary-set(inventory.columns)}"
inventory["acquisition_utc"] = pd.to_datetime(inventory.acquisition_utc,utc=True,errors="raise")
inventory["relative_orbit"] = pd.to_numeric(inventory.relative_orbit,errors="raise").astype(int)
inventory["year"] = inventory.acquisition_utc.dt.year
inventory["month"] = inventory.acquisition_utc.dt.month
inventory = inventory.loc[inventory.year.isin(YEARS) &
                          inventory.month.between(*MONTHS)].copy()
inventory = inventory.loc[[tuple(x) in TRACKS for x in
                           zip(inventory["pass"],inventory.relative_orbit)]]
if inventory.scene_id.duplicated().any():
    raise RuntimeError("Duplicate source scene IDs")
inventory["acquisition_local"] = inventory.acquisition_utc.dt.tz_convert("Asia/Dhaka")
inventory = inventory.sort_values("acquisition_utc").reset_index(drop=True)

s1 = (ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(ROI)
      .filter(ee.Filter.eq("instrumentMode","IW"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation","VV"))
      .filter(ee.Filter.listContains("transmitterReceiverPolarisation","VH")))
def selected_collection(year, p, orbit):
    return (s1.filterDate(f"{year}-{MONTHS[0]:02d}-01",f"{year}-{MONTHS[1]+1:02d}-01")
            .filter(ee.Filter.eq("orbitProperties_pass",p))
            .filter(ee.Filter.eq("relativeOrbitNumber_start",orbit)))
checks = []
for year in YEARS:
    for p,orbit in TRACKS:
        catalog = selected_collection(year,p,orbit)
        n = int(catalog.size().getInfo())
        if n > 100:
            raise RuntimeError("Unexpected annual scene count; check Earth Engine filters")
        remote = set(catalog.aggregate_array("system:index").getInfo())
        local = set(inventory.loc[(inventory.year==year)&(inventory["pass"]==p)&
                                  (inventory.relative_orbit==orbit),"scene_id"])
        if len(remote)!=n or remote!=local:
            raise RuntimeError(f"EE/inventory mismatch for {year} {p}/{orbit}. "
                               f"Remote-only: {list(remote-local)[:3]}, local-only: {list(local-remote)[:3]}")
        checks.append({"year":year,"orbit_pass":p,"relative_orbit":orbit,"scene_count":n})
checks = pd.DataFrame(checks)
checks.to_csv(TABLES/"scene12_catalog_inventory_check.csv",index=False)
print("Catalog and Notebook 09 scene IDs match for the analysis window.")
display(checks)
display(inventory.loc[(inventory.year==2022)&inventory.month.isin([6,7]),
                  ["scene_id","acquisition_utc","acquisition_local","pass","relative_orbit"]])


## 12C — Compute per-scene VV screening extents, no monthly median

Use the original incidence-angle-masked VV support after focal filtering. All area numerators and support denominators use the same projected 30 m pixel-area reduction. Processing proceeds in small batches and maintains a configuration-specific local cache. VH remains available for later distribution QA; no uncalibrated VH classification is asserted.

In [ ]:
def prepared_polarizations(image):
    angle = image.select("angle")
    angle_ok = angle.gte(ANGLE_RANGE[0]).And(angle.lte(ANGLE_RANGE[1]))
    vv_original = image.select("VV").updateMask(angle_ok)
    vh_original = image.select("VH").updateMask(angle_ok)
    vv = vv_original.focal_median(radius=FILTER_RADIUS_M,kernelType="circle",units="meters")
    vh = vh_original.focal_median(radius=FILTER_RADIUS_M,kernelType="circle",units="meters")
    return (vv.updateMask(vv_original.mask()).rename("VV"),
            vh.updateMask(vh_original.mask()).rename("VH"))

def summarize_scene(scene_id):
    image = ee.Image("COPERNICUS/S1_GRD/"+scene_id)
    vv, vh = prepared_polarizations(image)
    observed = vv.mask().unmask(0).gt(0)
    pixel = ee.Image.pixelArea()
    bands = [pixel.multiply(observed).rename("observed_m2")]
    for cutoff,label in zip(THRESHOLDS_DB,["m19","m17","m15"]):
        flagged = vv.lte(cutoff).unmask(0).And(observed)
        bands.append(pixel.multiply(flagged).rename("area_"+label+"_m2"))
    areas = ee.Image.cat(bands).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=ROI, crs="EPSG:32646",
        scale=SCALE_M, maxPixels=100_000_000, tileScale=4)
    return ee.Feature(None,areas).set("scene_id",scene_id)

chunks = []
for year in YEARS:
    for p,orbit in TRACKS:
        source = inventory.loc[(inventory.year==year)&
                               (inventory["pass"]==p)&
                               (inventory.relative_orbit==orbit)]
        ids = source.scene_id.tolist()
        cache = CACHE / f"S1_{year}_{p}_{orbit}_{CONFIG_HASH}.csv"
        if cache.is_file() and not FORCE_RECOMPUTE:
            output = pd.read_csv(cache)
            if set(output.scene_id)!=set(ids):
                raise RuntimeError("Cache scene IDs differ from the current inventory")
        else:
            print(f"Extracting {year} {p} / orbit {orbit}: {len(ids)} scenes")
            rows = []
            for k in range(0,len(ids),8):
                group = [summarize_scene(sid) for sid in ids[k:k+8]]
                if group:
                    rows.extend(x["properties"] for x in ee.FeatureCollection(group).getInfo()["features"])
            output = pd.DataFrame(rows) if rows else pd.DataFrame(
                columns=["scene_id","observed_m2","area_m19_m2","area_m17_m2","area_m15_m2"])
            if set(output.scene_id)!=set(ids):
                raise RuntimeError("Server did not return all expected scenes")
            output.to_csv(cache,index=False)
        chunks.append(output)
area_data = pd.concat(chunks,ignore_index=True)
if area_data.scene_id.duplicated().any():
    raise RuntimeError("Duplicated Earth Engine outputs")
scenes = inventory.merge(area_data,on="scene_id",how="left",validate="one_to_one")
for field in ["observed_m2","area_m19_m2","area_m17_m2","area_m15_m2"]:
    scenes[field] = pd.to_numeric(scenes[field],errors="raise")
scenes["observed_km2"] = scenes.observed_m2 / 1e6
for key in ("m19","m17","m15"):
    scenes["lowvv_"+key+"_km2"] = scenes["area_"+key+"_m2"]/1e6
scenes["observed_fraction"] = scenes.observed_km2 / REFERENCE_KM2
scenes["eligible"] = scenes.observed_fraction.ge(MIN_SCENE_COVERAGE)
scenes["screened_fraction_of_observed"] = (
    scenes.lowvv_m17_km2 / scenes.observed_km2.replace(0,np.nan))
scenes["config_hash"] = CONFIG_HASH
check_scene_areas(scenes, REFERENCE_KM2)
if not scenes.observed_fraction.between(0,1.001).all():
    raise RuntimeError("Scene support fraction is outside accepted range")
scene_csv = TABLES/"scene12_S1_per_acquisition_2022_2025.csv"
scenes.to_csv(scene_csv,index=False)
scene_summary = describe_scene_support(scenes)
scene_summary.to_csv(TABLES/"scene12_pass_support_summary.csv",index=False)
print("Extracted scenes:",len(scenes),"| meeting 90% support:",int(scenes.eligible.sum()))
display(scene_summary)
